# Dimensionality Reduction: Choosing the Right Method

**Topic:** Unsupervised Learning — Dimensionality Reduction

In [ ]:
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, FloatSlider, Output, HBox, VBox
from IPython.display import display, clear_output, Markdown
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print('umap-learn not installed. Run: pip install umap-learn')

if UMAP_AVAILABLE:
    # Warm up UMAP's JIT compilation on a tiny dummy array so the speed
    # comparison widget's first render isn't skewed by one-time compile cost.
    umap.UMAP(n_neighbors=5, n_components=2).fit_transform(np.random.rand(50, 4))

from tkh_utils import PALETTE, FONT, base_layout, load_california_housing


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the key differences between PCA, t-SNE, and UMAP along four dimensions: speed, interpretability, local structure, global structure
- **Explain** when to reach for each method on a real dataset
- **Interpret** side-by-side projections and recognize what each algorithm reveals and hides

> **Tip:** In the comparison widget, select each algorithm and notice what changes. PCA axes have units (principal components). t-SNE and UMAP axes have no units — only the positions of points relative to each other matter.

---
## How we got here

You have now studied all three major linear and non-linear dimensionality reduction methods:
- **[07_pca.ipynb](07_pca.ipynb)** — linear, fast, interpretable, preserves variance
- **[08_tsne.ipynb](08_tsne.ipynb)** — non-linear, slow, excellent local structure
- **[09_umap.ipynb](09_umap.ipynb)** — non-linear, faster, better global structure

This notebook applies all three to the California housing dataset and builds your judgment for which to use when.

---
## Why this matters for data science

Choosing the wrong dimensionality reduction method can lead you to incorrect conclusions. A PCA projection that looks like two overlapping clouds may actually contain several well-separated groups that t-SNE or UMAP would reveal. Conversely, a t-SNE plot that looks like 20 tiny clusters may represent a single continuous manifold that UMAP would show more faithfully.

Understanding the assumptions and failure modes of each method is how you avoid being misled by your own visualization.

In **[ml_concepts/11_the_curse_of_dimensionality.ipynb](../ml_concepts/11_the_curse_of_dimensionality.ipynb)** you learned why raw distances stop being trustworthy in high dimensions — that's the shared problem all three of these methods are solving, just with different trade-offs.

\
---
## Where it sits on the spectrum

Referencing **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)**:

| Method | Interpretability | Complexity | Global structure | Local structure | Speed |
|--------|-----------------|------------|-----------------|-----------------|-------|
| PCA | High (axes = PCs) | Low | Excellent | Poor (linear) | Fast |
| t-SNE | Low (axes = arbitrary) | High | Poor | Excellent | Slow |
| UMAP | Low (axes = arbitrary) | Medium | Good | Excellent | Medium |

---
## How it learns

Each algorithm answers a different question about the data.

**PCA** asks: *What linear directions explain the most variance?* It finds orthogonal axes along which the data is most spread and projects onto them. Fast, deterministic, but only linear.

**t-SNE** asks: *Which points are neighbors in high dimensions, and can I preserve those neighborhoods in 2D?* It optimizes a 2D layout to match high-dimensional neighborhood probabilities. Non-linear, revealing, but slow and only for visualization.

**UMAP** asks: *What graph best represents the data's topology, and what 2D layout preserves that graph's structure?* Similar goal to t-SNE but with a different mathematical foundation that better preserves global structure and scales to larger datasets.

\
---
## The math behind it

| | PCA | t-SNE | UMAP |
|---|---|---|---|
| Foundation | Eigendecomposition of covariance | KL divergence minimization | Cross-entropy on fuzzy graph |
| Similarity measure | Linear (dot product) | Gaussian in high-dim, Student-t in low-dim | Exponential decay with local normalization |
| Optimization | Closed-form (SVD) | Gradient descent | Gradient descent |
| Deterministic? | Yes | Yes (fixed seed) | Yes (fixed seed) |
| Projects new data? | Yes | No | Yes (transform mode) |

---
## Try it yourself

In [ ]:
out1 = Output()

algo_dropdown = Dropdown(
    options=[("PCA", "pca"), ("t-SNE", "tsne"), ("UMAP", "umap")],
    value="pca",
    description="Algorithm:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
param_slider = IntSlider(
    value=30, min=5, max=50, step=5,
    description="Perplexity / n_neighbors:",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="420px"),
)

X, y = load_california_housing()
X_sc = StandardScaler().fit_transform(X)
price_tier = pd.qcut(y, q=3, labels=["Low", "Mid", "High"])

rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_sc), size=1500, replace=False)
X_sample = X_sc[sample_idx]
tier_sample = price_tier.iloc[sample_idx].reset_index(drop=True)

tier_colors = {"Low": PALETTE["primary"], "Mid": PALETTE["accent"], "High": PALETTE["secondary"]}

def render_dashboard(algo, param):
    param_slider.disabled = (algo == "pca")

    if algo == "pca":
        Z = PCA(n_components=2, random_state=42).fit_transform(X_sample)
        title = "PCA 2D Projection (no parameter — n_components=2 is fixed)"
        caption = "**PCA projection — no parameter to tune.** `n_components=2` is fixed, so this panel never changes; it's the fast, deterministic baseline the other two are compared against."
    elif algo == "tsne":
        Z = TSNE(n_components=2, perplexity=param, random_state=42, init="pca").fit_transform(X_sample)
        title = f"t-SNE 2D Projection (perplexity={param})"
        caption = f"**t-SNE projection — perplexity={param}.** Recomputed from scratch at this value; lower perplexity fragments the price tiers into tighter sub-clusters, higher perplexity smooths them together."
    else:
        if UMAP_AVAILABLE:
            Z = umap.UMAP(n_components=2, n_neighbors=param, min_dist=0.1, random_state=42).fit_transform(X_sample)
            title = f"UMAP 2D Projection (n_neighbors={param})"
            caption = f"**UMAP projection — n_neighbors={param}.** Recomputed from scratch at this value; lower n_neighbors emphasizes local structure, higher n_neighbors emphasizes global shape."
        else:
            Z = TSNE(n_components=2, perplexity=30, random_state=42, init="pca").fit_transform(X_sample)
            title = "UMAP unavailable — showing t-SNE instead (pip install umap-learn)"
            caption = "**UMAP unavailable.** Showing t-SNE (perplexity=30) instead — run `pip install umap-learn` to see the real UMAP projection here."

    traces = []
    for tier, color in tier_colors.items():
        mask = (tier_sample == tier).to_numpy()
        traces.append(go.Scatter(
            x=Z[mask, 0], y=Z[mask, 1], mode="markers",
            marker=dict(color=color, size=5, opacity=0.6),
            name=f"{tier} price",
        ))
    layout = base_layout(title=title, xaxis_title="Dimension 1", yaxis_title="Dimension 2")
    fig = go.Figure(data=traces, layout=layout)
    with out1:
        clear_output(wait=True)
        fig.show()
        display(Markdown(caption))

def on_change_dashboard(change):
    render_dashboard(algo_dropdown.value, param_slider.value)

algo_dropdown.observe(on_change_dashboard, names="value")
param_slider.observe(on_change_dashboard, names="value")
display(VBox([algo_dropdown, param_slider, out1]))
render_dashboard(algo_dropdown.value, param_slider.value)

In [ ]:
out2 = Output()

size_slider = IntSlider(
    value=1000, min=500, max=3000, step=500,
    description="Sample size:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"),
)

def render_speed(n):
    idx = rng.choice(len(X_sc), size=n, replace=False)
    X_n = X_sc[idx]

    t0 = time.perf_counter()
    PCA(n_components=2, random_state=42).fit_transform(X_n)
    pca_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    TSNE(n_components=2, perplexity=30, random_state=42, init="pca").fit_transform(X_n)
    tsne_time = time.perf_counter() - t0

    if UMAP_AVAILABLE:
        t0 = time.perf_counter()
        umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_n)
        umap_time = time.perf_counter() - t0
        umap_label = "UMAP"
    else:
        umap_time = 0
        umap_label = "UMAP (not installed)"

    labels = ["PCA", "t-SNE", umap_label]
    times = [pca_time, tsne_time, umap_time]
    colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"]]

    fig = go.Figure([go.Bar(
        x=labels, y=times, marker_color=colors,
        text=[f"{t:.2f}s" if t else "n/a" for t in times],
        textposition="outside",
    )])
    layout = base_layout(
        title=f"Fit Time on {n:,} Samples",
        xaxis_title="Method",
        yaxis_title="Fit time (seconds)",
    )
    fig.update_layout(layout)

    if UMAP_AVAILABLE:
        outcome = "UMAP edges out t-SNE at this size" if umap_time < tsne_time else "UMAP does not beat t-SNE at this size"
        caption = (
            f"**At n={n:,}: PCA={pca_time:.3f}s, t-SNE={tsne_time:.3f}s, UMAP={umap_time:.3f}s.** "
            f"{outcome} — its speed advantage is asymptotic, typically appearing only at far larger "
            f"sample sizes than this slider reaches, and fixing `random_state` here (for a reproducible "
            f"demo) disables the parallelism UMAP normally relies on."
        )
    else:
        caption = (
            f"**At n={n:,}: PCA={pca_time:.3f}s, t-SNE={tsne_time:.3f}s.** "
            f"UMAP isn't installed in this environment — install `umap-learn` to see its bar."
        )

    with out2:
        clear_output(wait=True)
        fig.show()
        display(Markdown(caption))

def on_change_speed(change):
    render_speed(size_slider.value)

size_slider.observe(on_change_speed, names="value")
display(VBox([size_slider, out2]))
render_speed(size_slider.value)

---
## What's happening?

The dashboard widget shows that the three methods reveal different aspects of the same data. PCA's linear projection can miss non-linear structure — price tiers may overlap in PCA space even though they're distinguishable in the original 8-dimensional feature space. Switch to t-SNE or UMAP and drag their slider: both tend to pull the tiers apart more clearly than PCA does.

The speed comparison widget makes the scalability gap concrete: drag the sample-size slider up and watch PCA's bar barely move while t-SNE's climbs steeply — this is exactly the "PCA is dramatically faster" claim from the strengths-and-weaknesses table below, but now as a live measurement instead of an assertion.

---
## Key hyperparameters

**PCA**: `n_components` (keep enough for 90%+ variance) — no stochastic element, always fast.

**t-SNE**: `perplexity` (15-50 for most datasets), `max_iter` (increase for convergence), `random_state` for reproducibility.

**UMAP**: `n_neighbors` (15-30 typical), `min_dist` (0.1 default), `random_state` for reproducibility.

**Rule of thumb**: Start with PCA. If PCA shows clear separation, stop. If not, try UMAP (it's faster). Only reach for t-SNE if UMAP misses important local structure.

---
## Strengths and weaknesses

| | PCA | t-SNE | UMAP |
|---|---|---|---|
| Speed | ★★★ Fast | ★ Slow | ★★ Medium |
| Scalability | ★★★ Millions | ★ <100k | ★★★ Millions |
| Global structure | ★★★ Excellent | ★ Poor | ★★ Good |
| Local structure | ★ Linear only | ★★★ Excellent | ★★★ Excellent |
| Interpretability | ★★★ High | ★ None | ★ None |
| Projects new data | ★★★ Yes | ★ No | ★★★ Yes |

---
## When to use each

| Goal | Method |
|------|--------|
| Quick exploratory visualization | PCA first |
| Reveal non-linear cluster structure | UMAP (faster) or t-SNE (if UMAP unavailable) |
| Preserve global cluster distances | UMAP |
| Maximum local cluster separation | t-SNE |
| Feature engineering (not just viz) | PCA or UMAP |
| Project new data without rerunning | PCA or UMAP |

---
## Key takeaway

> **PCA is the fast starting point, t-SNE excels at revealing local cluster structure, and UMAP balances speed with global structure preservation — start with PCA and escalate only when the linear view is insufficient.**

---
*Next up: 11_anomaly_detection — where you find the data points that don't fit anywhere*